# Mục 3b(iv) — Xử lý nhãn DEATH_EVENT không hợp lệ
**Võ Minh Huy — 2311110117**

Miền mã hợp lệ là 0/1. Nhãn 2 chưa biết giá trị đúng nên chuyển thành thiếu trên bản sao, giữ hồ sơ chờ xác minh. Không đoán 0/1, không dùng mode hay tra nhãn từ dữ liệu original.

## 1. Đọc dữ liệu và xử lý mã ngoài miền
Giữ đủ 299 hồ sơ trong bản làm việc. Tách tập có nhãn hợp lệ và tập chờ xác minh.

In [1]:
from pathlib import Path
from decimal import Decimal
from collections import Counter
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_vao = thu_muc / "ket_qua/11_dien_ejection_fraction/ban_lam_viec_sau_ef_python.csv"
ban_dau = pd.read_csv(file_vao, dtype=str, keep_default_na=False)
ban_dau["MaHoSo"] = ban_dau["MaHoSo"].astype(int)
ban_dau["DEATH_EVENT"] = ban_dau["DEATH_EVENT"].map(
    lambda v: Decimal(v) if v.strip() else None)
ban_nhan = ban_dau.copy(deep=True)
mat_na_hop_le = ban_dau["DEATH_EVENT"].isin([0, 1])
nhat_ky = ban_dau.loc[~mat_na_hop_le, ["MaHoSo", "DEATH_EVENT"]].rename(
    columns={"DEATH_EVENT":"NhanTruoc"}).copy()
nhat_ky["LyDo"] = nhat_ky["NhanTruoc"].map(
    lambda v: "Nhãn bị thiếu" if pd.isna(v) else "Nhãn ngoài miền 0/1")
ban_nhan.loc[~mat_na_hop_le, "DEATH_EVENT"] = None
nhat_ky["NhanSau"] = pd.array([None] * len(nhat_ky), dtype="Int64")
nhat_ky["CachXuLy"] = "Giữ hồ sơ; chờ xác minh nhãn"
nhat_ky = nhat_ky[["MaHoSo", "NhanTruoc", "NhanSau", "LyDo", "CachXuLy"]]
nhan_hop_le = ban_nhan.loc[ban_nhan["DEATH_EVENT"].isin([0, 1])].copy()
cho_xac_minh = ban_nhan.loc[ban_nhan["DEATH_EVENT"].isna()].copy()

tong_ket = pd.DataFrame({"SoHoSoTruoc":[len(ban_dau)],
    "SoHoSoGiuLai":[len(ban_nhan)],
    "SoNhanSai":[(ban_dau["DEATH_EVENT"].notna() & ~mat_na_hop_le).sum()],
    "SoNhanHopLe":[len(nhan_hop_le)], "SoChoXacMinh":[len(cho_xac_minh)],
    "SoNhanSaiSauXuLy":[(ban_nhan["DEATH_EVENT"].notna() &
                         ~ban_nhan["DEATH_EVENT"].isin([0, 1])).sum()]})
tong_ket

,SoHoSoTruoc,SoHoSoGiuLai,SoNhanSai,SoNhanHopLe,SoChoXacMinh,SoNhanSaiSauXuLy
0,299,299,1,298,1,0


## 2. Nhật ký nhãn trước sau
<NA> trong Python tương ứng NULL trong SQL. MaHoSo đánh số nội bộ có thể khác SQL; không dùng số này để suy ra hai hồ sơ khác nhau.

In [2]:
nhat_ky

,MaHoSo,NhanTruoc,NhanSau,LyDo,CachXuLy
60,61,2.000000,<NA>,Nhãn ngoài miền 0/1,Giữ hồ sơ; chờ xác minh nhãn


## 3. Tần suất trước sau
Kiểm tra nhãn 0 và 1 giữ nguyên số lượng; chỉ mã sai 2 chuyển thành thiếu.

In [3]:
dong_tan_suat = []
for giai_doan, bang in [("Trước xử lý", ban_dau), ("Sau xử lý", ban_nhan)]:
    dem = Counter(bang["DEATH_EVENT"].tolist())
    for nhan, so_luong in sorted(dem.items(), key=lambda item: (
            item[0] is not None, item[0] if item[0] is not None else 0)):
        dong_tan_suat.append({"GiaiDoan":giai_doan,
            "Nhan":"<NULL>" if nhan is None else str(int(nhan)),
            "SoLuong":so_luong})
tan_suat = pd.DataFrame(dong_tan_suat)
tan_suat

,GiaiDoan,Nhan,SoLuong
0,Trước xử lý,0,203
1,Trước xử lý,1,95
2,Trước xử lý,2,1
3,Sau xử lý,<NULL>,1
4,Sau xử lý,0,203
5,Sau xử lý,1,95


**Nhận xét:** Giữ 299 hồ sơ: 298 có nhãn hợp lệ (203 nhãn 0, 95 nhãn 1), một hồ sơ chờ xác minh. Chuyển 2 thành thiếu chưa khôi phục được nhãn thật. Chỉ dùng tập 298 hồ sơ cho phân tích cần nhãn. Các thuộc tính khác và dữ liệu nguồn giữ nguyên.

## 4. Lưu kết quả trung gian
Lưu tổng kết, nhật ký, tần suất, bản giữ đủ hồ sơ và hai tập tách riêng. Chưa tạo after cuối.

In [4]:
thu_muc_kq = thu_muc / "ket_qua/15_khong_nhat_quan/02_death_event"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
for ten, bang in [("tong_ket_python.csv", tong_ket), ("nhat_ky_python.csv", nhat_ky),
                  ("tan_suat_python.csv", tan_suat), ("ban_lam_viec_nhan_python.csv", ban_nhan),
                  ("nhan_hop_le_python.csv", nhan_hop_le), ("cho_xac_minh_python.csv", cho_xac_minh)]:
    bang.to_csv(thu_muc_kq / ten, index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu kết quả; giữ nguyên CSV đầu vào."]})

,TrangThai
0,Đã lưu kết quả; giữ nguyên CSV đầu vào.
